In [3]:
import pandas as pd

movies = pd.read_csv("../data/movies.csv")
ratings = pd.read_csv("../data/ratings.csv")
tags = pd.read_csv("../data/tags.csv")

print("Movies:", movies.shape)
print("Ratings:", ratings.shape)
print("Tags:", tags.shape)

display(movies.head())
display(ratings.head())
display(tags.head())

Movies: (9742, 3)
Ratings: (100836, 4)
Tags: (3683, 4)


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


,userId,movieId,tag,timestamp
0,2,60756,funny,1445714994
1,2,60756,Highly quotable,1445714996
2,2,60756,will ferrell,1445714992
3,2,89774,Boxing story,1445715207
4,2,89774,MMA,1445715200


In [4]:
print("Movies:", movies.shape)
print("Ratings:", ratings.shape)
print("Tags:", tags.shape)
print("Links:", pd.read_csv("../data/links.csv").shape)

Movies: (9742, 3)
Ratings: (100836, 4)
Tags: (3683, 4)
Links: (9742, 3)


In [5]:
tag_data = tags.groupby("movieId")["tag"].apply(
    lambda x: " ".join(x.astype(str))
).reset_index()

movies_text = movies.merge(
    tag_data,
    on="movieId",
    how="left"
)
movies_text["tag"] = movies_text["tag"].fillna("")

movies_text["text"] = (
    movies_text["title"] + " "
    + movies_text["genres"].str.replace("|", " ", regex=False) + " "
    + movies_text["tag"]
)

movies_text.head()

,movieId,title,genres,tag,text
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,pixar pixar fun,Toy Story (1995) Adventure Animation Children ...
1,2,Jumanji (1995),Adventure|Children|Fantasy,fantasy magic board game Robin Williams game,Jumanji (1995) Adventure Children Fantasy fant...
2,3,Grumpier Old Men (1995),Comedy|Romance,moldy old,Grumpier Old Men (1995) Comedy Romance moldy old
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,,Waiting to Exhale (1995) Comedy Drama Romance
4,5,Father of the Bride Part II (1995),Comedy,pregnancy remake,Father of the Bride Part II (1995) Comedy preg...


In [6]:
print("Missing titles:", movies_text["title"].isna().sum())
print("Missing genres:", movies_text["genres"].isna().sum())
print("Missing text:", movies_text["text"].isna().sum())

Missing titles: 0
Missing genres: 0
Missing text: 0


In [7]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "sentence-transformers/all-mpnet-base-v2"
)

movie_embeddings = embedding_model.encode(
    movies_text["text"].tolist(),
    show_progress_bar=True
)

print("Embedding shape:", movie_embeddings.shape)

c:\Users\MEM0\CineMate\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Batches: 100%|██████████| 305/305 [04:29<00:00,  1.13it/s]


Embedding shape: (9742, 768)


In [8]:
from sklearn.metrics.pairwise import cosine_similarity

def recommend_movies(query, top_n=10):
    query_embedding = embedding_model.encode([query])

    similarities = cosine_similarity(
        query_embedding,
        movie_embeddings
    )[0]

    top_indices = similarities.argsort()[-top_n:][::-1]

    recommendations = movies_text.iloc[top_indices].copy()
    recommendations["similarity"] = similarities[top_indices]

    return recommendations[["title", "genres", "similarity"]]

In [9]:
recommend_movies(
    "I want a serious, intelligent science fiction movie with space, mystery and a deep story.",
    top_n=10
)

,title,genres,similarity
8376,Interstellar (2014),Sci-Fi|IMAX,0.597852
8159,Star Trek Into Darkness (2013),Action|Adventure|Sci-Fi|IMAX,0.595216
6199,Mission: Impossible III (2006),Action|Adventure|Thriller,0.577462
9485,Alien: Covenant (2017),Action|Horror|Sci-Fi|Thriller,0.563975
8917,Star Trek Beyond (2016),Action|Adventure|Sci-Fi,0.560661
8697,Doctor Strange (2016),Action|Adventure|Sci-Fi,0.559982
8439,Mission: Impossible - Rogue Nation (2015),Action|Adventure|Thriller,0.555850
8879,The Martian (2015),Adventure|Drama|Sci-Fi,0.555447
7774,Mission: Impossible - Ghost Protocol (2011),Action|Adventure|Thriller|IMAX,0.551650
8252,Gravity (2013),Action|Sci-Fi|IMAX,0.551342


In [10]:

rating_stats = ratings.groupby("movieId")["rating"].agg(
    ["mean", "count"]
).reset_index()

rating_stats.columns = ["movieId", "avg_rating", "rating_count"]

movies_text = movies_text.merge(
    rating_stats,
    on="movieId",
    how="left"
)

movies_text.head()

,movieId,title,genres,tag,text,avg_rating,rating_count
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,pixar pixar fun,Toy Story (1995) Adventure Animation Children ...,3.920930,215.0
1,2,Jumanji (1995),Adventure|Children|Fantasy,fantasy magic board game Robin Williams game,Jumanji (1995) Adventure Children Fantasy fant...,3.431818,110.0
2,3,Grumpier Old Men (1995),Comedy|Romance,moldy old,Grumpier Old Men (1995) Comedy Romance moldy old,3.259615,52.0
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,,Waiting to Exhale (1995) Comedy Drama Romance,2.357143,7.0
4,5,Father of the Bride Part II (1995),Comedy,pregnancy remake,Father of the Bride Part II (1995) Comedy preg...,3.071429,49.0


In [11]:
m = 50

C = ratings["rating"].mean()

movies_text["quality_score"] = (
    (movies_text["rating_count"] / (movies_text["rating_count"] + m))
    * movies_text["avg_rating"]
    +
    (m / (movies_text["rating_count"] + m))
    * C
)

movies_text[[
    "title",
    "avg_rating",
    "rating_count",
    "quality_score"
]].sort_values(
    "quality_score",
    ascending=False
).head(10)

,title,avg_rating,rating_count,quality_score
277,"Shawshank Redemption, The (1994)",4.429022,317.0,4.302664
2226,Fight Club (1999),4.272936,218.0,4.129022
659,"Godfather, The (1972)",4.289062,192.0,4.126355
224,Star Wars: Episode IV - A New Hope (1977),4.231076,251.0,4.109893
257,Pulp Fiction (1994),4.197068,307.0,4.099658
46,"Usual Suspects, The (1995)",4.237745,204.0,4.092826
461,Schindler's List (1993),4.225000,220.0,4.091029
1939,"Matrix, The (1999)",4.192446,278.0,4.087128
898,Star Wars: Episode V - The Empire Strikes Back...,4.215640,211.0,4.078842
314,Forrest Gump (1994),4.164134,329.0,4.076723


In [12]:
from scipy.sparse import csr_matrix
from sklearn.decomposition import TruncatedSVD

In [13]:
user_movie_matrix = ratings.pivot_table(
    index="userId",
    columns="movieId",
    values="rating",
    fill_value=0
)

print("User-movie matrix:", user_movie_matrix.shape)

User-movie matrix: (610, 9724)


In [14]:
svd = TruncatedSVD(
    n_components=50,
    random_state=42
)

user_factors = svd.fit_transform(user_movie_matrix)

movie_factors = svd.components_.T

print("User factors:", user_factors.shape)
print("Movie factors:", movie_factors.shape)

User factors: (610, 50)
Movie factors: (9724, 50)


In [15]:
import numpy as np

In [16]:
movie_id_to_index = {
    movie_id: index
    for index, movie_id in enumerate(user_movie_matrix.columns)
}

In [17]:
def collaborative_recommend(user_id, top_n=10):
    user_index = user_movie_matrix.index.get_loc(user_id)

    predicted_scores = user_factors[user_index] @ movie_factors.T

    watched_movies = set(
        ratings.loc[ratings["userId"] == user_id, "movieId"]
    )

    for movie_id in watched_movies:
        if movie_id in movie_id_to_index:
            predicted_scores[movie_id_to_index[movie_id]] = -np.inf

    top_indices = np.argsort(predicted_scores)[-top_n:][::-1]

    recommendations = movies_text[
        movies_text["movieId"].isin(
            user_movie_matrix.columns[top_indices]
        )
    ].copy()

    order = {
        movie_id: rank
        for rank, movie_id in enumerate(
            user_movie_matrix.columns[top_indices]
        )
    }

    recommendations["rank"] = recommendations["movieId"].map(order)

    return recommendations.sort_values("rank")[
        ["movieId", "title", "genres"]
    ]

In [18]:
collaborative_recommend(user_id=1, top_n=10)

,movieId,title,genres
793,1036,Die Hard (1988),Action|Crime|Thriller
922,1221,"Godfather: Part II, The (1974)",Crime|Drama
659,858,"Godfather, The (1972)",Crime|Drama
958,1259,Stand by Me (1986),Adventure|Drama
1067,1387,Jaws (1975),Action|Horror
507,589,Terminator 2: Judgment Day (1991),Action|Sci-Fi
1445,1968,"Breakfast Club, The (1985)",Comedy|Drama
902,1200,Aliens (1986),Action|Adventure|Horror|Sci-Fi
2195,2918,Ferris Bueller's Day Off (1986),Comedy
2110,2804,"Christmas Story, A (1983)",Children|Comedy


In [19]:
def hybrid_recommend(user_id, query, top_n=10):
    query_embedding = embedding_model.encode([query])

    semantic_scores = cosine_similarity(
        query_embedding,
        movie_embeddings
    )[0]

    user_index = user_movie_matrix.index.get_loc(user_id)

    collaborative_scores = user_factors[user_index] @ movie_factors.T

    collaborative_scores = (
        collaborative_scores - collaborative_scores.min()
    ) / (
        collaborative_scores.max() - collaborative_scores.min()
    )

    movie_ids = user_movie_matrix.columns

    hybrid_scores = (
        0.7 * semantic_scores[movie_ids.map(movie_id_to_index)]
        + 0.3 * collaborative_scores
    )

    watched_movies = set(
        ratings.loc[ratings["userId"] == user_id, "movieId"]
    )

    for movie_id in watched_movies:
        if movie_id in movie_id_to_index:
            hybrid_scores[movie_id_to_index[movie_id]] = -np.inf

    top_indices = np.argsort(hybrid_scores)[-top_n:][::-1]

    top_movie_ids = movie_ids[top_indices]

    recommendations = movies_text[
        movies_text["movieId"].isin(top_movie_ids)
    ].copy()

    order = {
        movie_id: rank
        for rank, movie_id in enumerate(top_movie_ids)
    }

    recommendations["rank"] = recommendations["movieId"].map(order)
    recommendations["score"] = recommendations["movieId"].map(
        lambda x: hybrid_scores[movie_id_to_index[x]]
    )

    return recommendations.sort_values("rank")[
        ["movieId", "title", "genres", "score"]
    ]

In [20]:
hybrid_recommend(
    user_id=1,
    query="I want a serious, intelligent science fiction movie with space, mystery and a deep story.",
    top_n=10
)

,movieId,title,genres,score
8177,102852,With Great Power: The Stan Lee Story (2012),Documentary,0.472236
8394,109971,Ocho apellidos vascos (2014),Comedy,0.470405
1059,1376,Star Trek IV: The Voyage Home (1986),Adventure|Comedy|Sci-Fi,0.457691
6216,45662,"Omen, The (2006)",Horror|Thriller,0.457246
903,1201,"Good, the Bad and the Ugly, The (Buono, il bru...",Action|Adventure|Western,0.454713
793,1036,Die Hard (1988),Action|Crime|Thriller,0.451767
9503,170875,The Fate of the Furious (2017),Action|Crime|Drama|Thriller,0.448924
1057,1374,Star Trek II: The Wrath of Khan (1982),Action|Adventure|Sci-Fi|Thriller,0.446549
8935,136355,Big Top Scooby-Doo! (2012),Animation|Children|Comedy,0.446460
8715,125970,Halloweentown (1998),Adventure|Children|Comedy|Fantasy,0.446137


In [21]:
def hybrid_recommend(user_id, query, top_n=10, genre=None):

    query_embedding = embedding_model.encode([query])

    semantic_scores = cosine_similarity(
        query_embedding,
        movie_embeddings
    )[0]

    user_index = user_movie_matrix.index.get_loc(user_id)

    collaborative_scores = user_factors[user_index] @ movie_factors.T

    collaborative_scores = (
        collaborative_scores - collaborative_scores.min()
    ) / (
        collaborative_scores.max() - collaborative_scores.min()
    )

    movie_ids = user_movie_matrix.columns

    hybrid_scores = (
        0.7 * semantic_scores[movie_ids.map(movie_id_to_index)]
        + 0.3 * collaborative_scores
    )

    watched_movies = set(
        ratings.loc[ratings["userId"] == user_id, "movieId"]
    )

    for movie_id in watched_movies:
        if movie_id in movie_id_to_index:
            hybrid_scores[movie_id_to_index[movie_id]] = -np.inf

    if genre:
        genre_mask = movies_text["genres"].str.contains(
            genre,
            case=False,
            na=False
        )

        allowed_movie_ids = set(
            movies_text.loc[genre_mask, "movieId"]
        )

        for i, movie_id in enumerate(movie_ids):
            if movie_id not in allowed_movie_ids:
                hybrid_scores[i] = -np.inf

    top_indices = np.argsort(hybrid_scores)[-top_n:][::-1]

    top_movie_ids = movie_ids[top_indices]

    recommendations = movies_text[
        movies_text["movieId"].isin(top_movie_ids)
    ].copy()

    order = {
        movie_id: rank
        for rank, movie_id in enumerate(top_movie_ids)
    }

    recommendations["rank"] = recommendations["movieId"].map(order)

    recommendations["score"] = recommendations["movieId"].map(
        lambda x: hybrid_scores[movie_id_to_index[x]]
    )

    return recommendations.sort_values("rank")[
        ["movieId", "title", "genres", "score"]
    ]

In [22]:
hybrid_recommend(
    user_id=1,
    query="I want a serious, intelligent science fiction movie with space, mystery and a deep story.",
    genre="Sci-Fi",
    top_n=10
)

,movieId,title,genres,score
1059,1376,Star Trek IV: The Voyage Home (1986),Adventure|Comedy|Sci-Fi,0.457691
1057,1374,Star Trek II: The Wrath of Khan (1982),Action|Adventure|Sci-Fi|Thriller,0.446549
1058,1375,Star Trek III: The Search for Spock (1984),Action|Adventure|Sci-Fi,0.444032
1055,1372,Star Trek VI: The Undiscovered Country (1991),Action|Mystery|Sci-Fi,0.437163
507,589,Terminator 2: Judgment Day (1991),Action|Sci-Fi,0.436134
9410,165343,The Rocky Horror Picture Show: Let's Do the Ti...,Comedy|Horror|Sci-Fi|Thriller,0.434918
8424,111360,Lucy (2014),Action|Sci-Fi,0.433223
8491,113350,I'll Follow You Down (2013),Drama|Mystery|Sci-Fi,0.429781
6238,46530,Superman Returns (2006),Action|Adventure|Sci-Fi|IMAX,0.421297
9731,191005,Gintama (2017),Action|Adventure|Comedy|Sci-Fi,0.417267


In [23]:
import os

print("API key found:", os.getenv("GEMINI_API_KEY") is not None)

API key found: True


In [2]:
from google import genai

client = genai.Client(
    api_key=os.getenv("GEMINI_API_KEY")
)

print("Gemini connected successfully!")

Gemini connected successfully!


In [26]:
def extract_preferences(user_message):
    prompt = f"""
You are the preference extraction system for a movie recommendation app.

Analyze the user's movie request and return ONLY valid JSON.

Use exactly these fields:
{{
    "genres": [],
    "moods": [],
    "themes": [],
    "exclude": []
}}

Rules:
- genres: movie genres explicitly requested or strongly implied
- moods: adjectives describing the desired feeling or tone
- themes: story elements, topics, settings, or concepts the user wants
- exclude: genres, themes, or types of movies the user does not want
- If something is not specified, return an empty list.
- Do not recommend movies.
- Do not explain anything.

User request:
{user_message}
"""

    response = client.models.generate_content(
        model="gemini-3.6-flash",
        contents=prompt
    )

    return response.text

In [27]:
extract_preferences(
    "I want a serious, intelligent science fiction movie with space, mystery and a deep story, but not superheroes."
)

'{\n    "genres": ["Science Fiction", "Mystery"],\n    "moods": ["serious", "intelligent"],\n    "themes": ["space", "deep story"],\n    "exclude": ["superheroes"]\n}'

In [28]:
genre_map = {
    "Science Fiction": "Sci-Fi",
    "Sci Fi": "Sci-Fi",
    "Science-Fiction": "Sci-Fi",
    "Thriller": "Thriller",
    "Mystery": "Mystery",
    "Action": "Action",
    "Adventure": "Adventure",
    "Comedy": "Comedy",
    "Drama": "Drama",
    "Horror": "Horror",
    "Romance": "Romance",
    "Fantasy": "Fantasy",
    "Animation": "Animation",
    "Children": "Children",
    "Crime": "Crime",
    "Documentary": "Documentary",
    "Musical": "Musical",
    "War": "War",
    "Western": "Western",
    "Film-Noir": "Film-Noir",
    "(no genres listed)": None
}

print("Genre mapping ready!")

Genre mapping ready!


In [29]:
genre="Sci-Fi"

In [30]:
import json

def get_preferences(user_message):
    response = extract_preferences(user_message)
    
    preferences = json.loads(response)

    preferences["genres"] = [
        genre_map[genre]
        for genre in preferences["genres"]
        if genre in genre_map and genre_map[genre] is not None
    ]

    return preferences

In [31]:
preferences = get_preferences(
    "I want a serious, intelligent science fiction movie with space, mystery and a deep story, but not superheroes."
)

preferences

{'genres': ['Sci-Fi', 'Mystery'],
 'moods': ['serious', 'intelligent'],
 'themes': ['space', 'deep story'],
 'exclude': ['superheroes']}

In [34]:
def filter_movies(preferences):
    candidates = movies_text.copy()

    # Every requested genre must be present
    for genre in preferences["genres"]:
        candidates = candidates[
            candidates["genres"].str.contains(
                genre,
                case=False,
                na=False
            )
        ]

    # Remove explicitly excluded types
    for excluded in preferences["exclude"]:
        candidates = candidates[
            ~candidates["text"].str.contains(
                excluded,
                case=False,
                na=False
            )
        ]

    return candidates

In [35]:
candidates = filter_movies(preferences)

print("Candidate movies:", len(candidates))
display(candidates[["title", "genres"]].head(10))

Candidate movies: 69


,title,genres
28,"City of Lost Children, The (Cité des enfants p...",Adventure|Drama|Fantasy|Mystery|Sci-Fi
31,Twelve Monkeys (a.k.a. 12 Monkeys) (1995),Mystery|Sci-Fi|Thriller
91,Unforgettable (1996),Mystery|Sci-Fi|Thriller
133,Congo (1995),Action|Adventure|Mystery|Sci-Fi
167,Strange Days (1995),Action|Crime|Drama|Mystery|Sci-Fi|Thriller
563,"Alphaville (Alphaville, une étrange aventure d...",Drama|Mystery|Romance|Sci-Fi|Thriller
932,Stalker (1979),Drama|Mystery|Sci-Fi
1055,Star Trek VI: The Undiscovered Country (1991),Action|Mystery|Sci-Fi
1391,"X-Files: Fight the Future, The (1998)",Action|Crime|Mystery|Sci-Fi|Thriller
1484,Soylent Green (1973),Drama|Mystery|Sci-Fi|Thriller


In [36]:
def rank_candidates(query, candidates, top_n=10):

    query_embedding = embedding_model.encode([query])

    candidate_indices = candidates.index
    candidate_embeddings = movie_embeddings[candidate_indices]

    scores = cosine_similarity(
        query_embedding,
        candidate_embeddings
    )[0]

    results = candidates.copy()
    results["semantic_score"] = scores

    return results.sort_values(
        "semantic_score",
        ascending=False
    ).head(top_n)[
        ["movieId", "title", "genres", "semantic_score"]
    ]

In [37]:
ranked = rank_candidates(
    "I want a serious, intelligent science fiction movie with space, mystery and a deep story, but not superheroes.",
    candidates,
    top_n=10
)

ranked

,movieId,title,genres,semantic_score
6797,60684,Watchmen (2009),Action|Drama|Mystery|Sci-Fi|Thriller|IMAX,0.489833
7014,68237,Moon (2009),Drama|Mystery|Sci-Fi|Thriller,0.469599
6802,60760,"X-Files: I Want to Believe, The (2008)",Drama|Mystery|Sci-Fi|Thriller,0.461456
7625,87306,Super 8 (2011),Mystery|Sci-Fi|Thriller|IMAX,0.456379
6664,57368,Cloverfield (2008),Action|Mystery|Sci-Fi|Thriller,0.433546
6331,48780,"Prestige, The (2006)",Drama|Mystery|Sci-Fi|Thriller,0.423659
4650,6949,"Big Empty, The (2003)",Comedy|Mystery|Sci-Fi,0.422312
7372,79132,Inception (2010),Action|Crime|Drama|Mystery|Sci-Fi|Thriller|IMAX,0.420322
1391,1909,"X-Files: Fight the Future, The (1998)",Action|Crime|Mystery|Sci-Fi|Thriller,0.419761
9689,184253,The Cloverfield Paradox (2018),Horror|Mystery|Sci-Fi|Thriller,0.417881


In [38]:
def hybrid_rank(user_id, query, candidates, top_n=10):

    query_embedding = embedding_model.encode([query])

    candidate_indices = candidates.index
    candidate_embeddings = movie_embeddings[candidate_indices]

    semantic_scores = cosine_similarity(
        query_embedding,
        candidate_embeddings
    )[0]

    user_index = user_movie_matrix.index.get_loc(user_id)

    collaborative_scores_all = (
        user_factors[user_index] @ movie_factors.T
    )

    collaborative_scores_all = (
        collaborative_scores_all - collaborative_scores_all.min()
    ) / (
        collaborative_scores_all.max()
        - collaborative_scores_all.min()
    )

    collaborative_scores = []

    for movie_id in candidates["movieId"]:
        if movie_id in movie_id_to_index:
            collaborative_scores.append(
                collaborative_scores_all[
                    movie_id_to_index[movie_id]
                ]
            )
        else:
            collaborative_scores.append(0)

    collaborative_scores = np.array(collaborative_scores)

    final_scores = (
        0.75 * semantic_scores
        + 0.25 * collaborative_scores
    )

    results = candidates.copy()
    results["semantic_score"] = semantic_scores
    results["collaborative_score"] = collaborative_scores
    results["final_score"] = final_scores

    return results.sort_values(
        "final_score",
        ascending=False
    ).head(top_n)[
        [
            "movieId",
            "title",
            "genres",
            "semantic_score",
            "collaborative_score",
            "final_score"
        ]
    ]

In [39]:
final_recommendations = hybrid_rank(
    user_id=1,
    query="I want a serious, intelligent science fiction movie with space, mystery and a deep story, but not superheroes.",
    candidates=candidates,
    top_n=10
)

final_recommendations

,movieId,title,genres,semantic_score,collaborative_score,final_score
6797,60684,Watchmen (2009),Action|Drama|Mystery|Sci-Fi|Thriller|IMAX,0.489833,0.141174,0.402668
7014,68237,Moon (2009),Drama|Mystery|Sci-Fi|Thriller,0.469599,0.191768,0.400141
6802,60760,"X-Files: I Want to Believe, The (2008)",Drama|Mystery|Sci-Fi|Thriller,0.461456,0.183489,0.391964
1055,1372,Star Trek VI: The Undiscovered Country (1991),Action|Mystery|Sci-Fi,0.412281,0.324075,0.390230
7625,87306,Super 8 (2011),Mystery|Sci-Fi|Thriller|IMAX,0.456379,0.184381,0.388380
1391,1909,"X-Files: Fight the Future, The (1998)",Action|Crime|Mystery|Sci-Fi|Thriller,0.419761,0.291645,0.387732
6664,57368,Cloverfield (2008),Action|Mystery|Sci-Fi|Thriller,0.433546,0.183581,0.371055
4650,6949,"Big Empty, The (2003)",Comedy|Mystery|Sci-Fi,0.422312,0.184135,0.362768
9668,182715,Annihilation (2018),Adventure|Mystery|Sci-Fi|Thriller,0.417044,0.195369,0.361625
9689,184253,The Cloverfield Paradox (2018),Horror|Mystery|Sci-Fi|Thriller,0.417881,0.192570,0.361553


In [40]:
print("Movie rows:", len(movies_text))
print("Embedding rows:", len(movie_embeddings))
print("Indexes match:", len(movies_text) == len(movie_embeddings))

Movie rows: 9742
Embedding rows: 9742
Indexes match: True


In [41]:
def mmr_rerank(candidates, movie_embeddings, top_n=10, diversity=0.3):
    selected = []

    candidate_indices = candidates.index.tolist()

    candidate_embeddings = movie_embeddings[candidate_indices]

    # Normalize embeddings
    candidate_embeddings = (
        candidate_embeddings
        / np.linalg.norm(
            candidate_embeddings,
            axis=1,
            keepdims=True
        )
    )

    first = candidates["final_score"].values.argmax()
    selected.append(first)

    while len(selected) < min(top_n, len(candidates)):

        remaining = [
            i for i in range(len(candidates))
            if i not in selected
        ]

        mmr_scores = []

        for i in remaining:

            relevance = candidates["final_score"].iloc[i]

            similarities = candidate_embeddings[i] @ (
                candidate_embeddings[selected].T
            )

            max_similarity = similarities.max()

            score = (
                (1 - diversity) * relevance
                - diversity * max_similarity
            )

            mmr_scores.append(score)

        best_index = remaining[np.argmax(mmr_scores)]

        selected.append(best_index)

    result = candidates.iloc[selected].copy()

    result["mmr_rank"] = range(1, len(result) + 1)

    return result[
        [
            "movieId",
            "title",
            "genres",
            "semantic_score",
            "collaborative_score",
            "final_score",
            "mmr_rank"
        ]
    ]

In [42]:
diverse_recommendations = mmr_rerank(
    final_recommendations,
    movie_embeddings,
    top_n=10,
    diversity=0.3
)

diverse_recommendations

,movieId,title,genres,semantic_score,collaborative_score,final_score,mmr_rank
6797,60684,Watchmen (2009),Action|Drama|Mystery|Sci-Fi|Thriller|IMAX,0.489833,0.141174,0.402668,1
1055,1372,Star Trek VI: The Undiscovered Country (1991),Action|Mystery|Sci-Fi,0.412281,0.324075,0.390230,2
4650,6949,"Big Empty, The (2003)",Comedy|Mystery|Sci-Fi,0.422312,0.184135,0.362768,3
1391,1909,"X-Files: Fight the Future, The (1998)",Action|Crime|Mystery|Sci-Fi|Thriller,0.419761,0.291645,0.387732,4
9668,182715,Annihilation (2018),Adventure|Mystery|Sci-Fi|Thriller,0.417044,0.195369,0.361625,5
7014,68237,Moon (2009),Drama|Mystery|Sci-Fi|Thriller,0.469599,0.191768,0.400141,6
7625,87306,Super 8 (2011),Mystery|Sci-Fi|Thriller|IMAX,0.456379,0.184381,0.388380,7
9689,184253,The Cloverfield Paradox (2018),Horror|Mystery|Sci-Fi|Thriller,0.417881,0.192570,0.361553,8
6802,60760,"X-Files: I Want to Believe, The (2008)",Drama|Mystery|Sci-Fi|Thriller,0.461456,0.183489,0.391964,9
6664,57368,Cloverfield (2008),Action|Mystery|Sci-Fi|Thriller,0.433546,0.183581,0.371055,10


In [43]:
top_candidates = hybrid_rank(
    user_id=1,
    query="I want a serious, intelligent science fiction movie with space, mystery and a deep story, but not superheroes.",
    candidates=candidates,
    top_n=30
)

print("Candidates for reranking:", len(top_candidates))
display(top_candidates[[
    "title",
    "genres",
    "final_score"
]])

Candidates for reranking: 30


,title,genres,final_score
6797,Watchmen (2009),Action|Drama|Mystery|Sci-Fi|Thriller|IMAX,0.402668
7014,Moon (2009),Drama|Mystery|Sci-Fi|Thriller,0.400141
6802,"X-Files: I Want to Believe, The (2008)",Drama|Mystery|Sci-Fi|Thriller,0.391964
1055,Star Trek VI: The Undiscovered Country (1991),Action|Mystery|Sci-Fi,0.390230
7625,Super 8 (2011),Mystery|Sci-Fi|Thriller|IMAX,0.388380
1391,"X-Files: Fight the Future, The (1998)",Action|Crime|Mystery|Sci-Fi|Thriller,0.387732
6664,Cloverfield (2008),Action|Mystery|Sci-Fi|Thriller,0.371055
4650,"Big Empty, The (2003)",Comedy|Mystery|Sci-Fi,0.362768
9668,Annihilation (2018),Adventure|Mystery|Sci-Fi|Thriller,0.361625
9689,The Cloverfield Paradox (2018),Horror|Mystery|Sci-Fi|Thriller,0.361553


In [44]:
diverse_recommendations = mmr_rerank(
    top_candidates,
    movie_embeddings,
    top_n=10,
    diversity=0.25
)

diverse_recommendations[
    ["title", "genres", "final_score", "mmr_rank"]
]

,title,genres,final_score,mmr_rank
6797,Watchmen (2009),Action|Drama|Mystery|Sci-Fi|Thriller|IMAX,0.402668,1
1055,Star Trek VI: The Undiscovered Country (1991),Action|Mystery|Sci-Fi,0.390230,2
1391,"X-Files: Fight the Future, The (1998)",Action|Crime|Mystery|Sci-Fi|Thriller,0.387732,3
4650,"Big Empty, The (2003)",Comedy|Mystery|Sci-Fi,0.362768,4
7014,Moon (2009),Drama|Mystery|Sci-Fi|Thriller,0.400141,5
6462,Aqua Teen Hunger Force Colon Movie Film for Th...,Action|Adventure|Animation|Comedy|Fantasy|Myst...,0.333293,6
9668,Annihilation (2018),Adventure|Mystery|Sci-Fi|Thriller,0.361625,7
4045,Galaxy of Terror (Quest) (1981),Action|Horror|Mystery|Sci-Fi,0.349429,8
3562,Donnie Darko (2001),Drama|Mystery|Sci-Fi|Thriller,0.325674,9
4631,Interstate 60 (2002),Adventure|Comedy|Drama|Fantasy|Mystery|Sci-Fi|...,0.337206,10


In [45]:
filtered_candidates = top_candidates[
    top_candidates["semantic_score"] >= 0.40
].copy()

print("Relevant candidates:", len(filtered_candidates))

Relevant candidates: 17


In [46]:
final_10 = mmr_rerank(
    filtered_candidates,
    movie_embeddings,
    top_n=10,
    diversity=0.10
)

final_10[[
    "title",
    "genres",
    "semantic_score",
    "final_score",
    "mmr_rank"
]]

,title,genres,semantic_score,final_score,mmr_rank
6797,Watchmen (2009),Action|Drama|Mystery|Sci-Fi|Thriller|IMAX,0.489833,0.402668,1
1055,Star Trek VI: The Undiscovered Country (1991),Action|Mystery|Sci-Fi,0.412281,0.390230,2
1391,"X-Files: Fight the Future, The (1998)",Action|Crime|Mystery|Sci-Fi|Thriller,0.419761,0.387732,3
7014,Moon (2009),Drama|Mystery|Sci-Fi|Thriller,0.469599,0.400141,4
7625,Super 8 (2011),Mystery|Sci-Fi|Thriller|IMAX,0.456379,0.388380,5
6802,"X-Files: I Want to Believe, The (2008)",Drama|Mystery|Sci-Fi|Thriller,0.461456,0.391964,6
9668,Annihilation (2018),Adventure|Mystery|Sci-Fi|Thriller,0.417044,0.361625,7
4650,"Big Empty, The (2003)",Comedy|Mystery|Sci-Fi,0.422312,0.362768,8
6664,Cloverfield (2008),Action|Mystery|Sci-Fi|Thriller,0.433546,0.371055,9
4045,Galaxy of Terror (Quest) (1981),Action|Horror|Mystery|Sci-Fi,0.405241,0.349429,10


In [47]:
print("Movies with tags:", tags["movieId"].nunique())
print("Total tags:", len(tags))

tag_counts = tags.groupby("movieId").size()

print("Average tags per tagged movie:", tag_counts.mean())
print("Maximum tags on one movie:", tag_counts.max())

Movies with tags: 1572
Total tags: 3683
Average tags per tagged movie: 2.3428753180661577
Maximum tags on one movie: 181


In [48]:
candidate_tags = tags[
    tags["movieId"].isin(candidates["movieId"])
]

display(
    candidate_tags[
        ["movieId", "tag"]
    ].sort_values("movieId").head(30)
)

,movieId,tag
998,29,kidnapping
696,32,time travel
2491,32,Post apocalyptic
2488,32,Brad Pitt
2492,32,post-apocalyptic
2493,32,remake
2494,32,time travel
2490,32,mindfuck
2489,32,Bruce Willis
2495,32,twist ending


In [49]:
def theme_match_score(preferences, candidates):

    requested_terms = (
        preferences["themes"]
        + preferences["moods"]
    )

    scores = []

    for movie_id in candidates["movieId"]:

        movie_tags = tags[
            tags["movieId"] == movie_id
        ]["tag"].astype(str).str.lower().tolist()

        tag_text = " ".join(movie_tags)

        matches = sum(
            term.lower() in tag_text
            for term in requested_terms
        )

        if requested_terms:
            score = matches / len(requested_terms)
        else:
            score = 0

        scores.append(score)

    return np.array(scores)

In [50]:
theme_scores = theme_match_score(
    preferences,
    candidates
)

print("Best theme match scores:")
print(sorted(theme_scores, reverse=True)[:10])

Best theme match scores:
[np.float64(0.25), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0)]


In [51]:
best_tag_movies = candidates.copy()

best_tag_movies["theme_score"] = theme_scores

display(
    best_tag_movies
    .sort_values("theme_score", ascending=False)
    [["movieId", "title", "genres", "theme_score"]]
    .head(10)
)

,movieId,title,genres,theme_score
7090,70286,District 9 (2009),Mystery|Sci-Fi|Thriller,0.25
31,32,Twelve Monkeys (a.k.a. 12 Monkeys) (1995),Mystery|Sci-Fi|Thriller,0.00
91,103,Unforgettable (1996),Mystery|Sci-Fi|Thriller,0.00
133,160,Congo (1995),Action|Adventure|Mystery|Sci-Fi,0.00
28,29,"City of Lost Children, The (Cité des enfants p...",Adventure|Drama|Fantasy|Mystery|Sci-Fi,0.00
167,198,Strange Days (1995),Action|Crime|Drama|Mystery|Sci-Fi|Thriller,0.00
563,680,"Alphaville (Alphaville, une étrange aventure d...",Drama|Mystery|Romance|Sci-Fi|Thriller,0.00
1055,1372,Star Trek VI: The Undiscovered Country (1991),Action|Mystery|Sci-Fi,0.00
932,1232,Stalker (1979),Drama|Mystery|Sci-Fi,0.00
1484,2009,Soylent Green (1973),Drama|Mystery|Sci-Fi|Thriller,0.00


In [52]:
movies_text["genre_text"] = (
    movies_text["genres"]
    .str.replace("|", ", ", regex=False)
)

In [53]:
movies_text["text"] = (
    "Movie title: " + movies_text["title"]
    + ". Genres: " + movies_text["genre_text"]
    + ". Tags: " + movies_text["tag"]
)

In [54]:
display(
    movies_text[
        ["title", "genres", "tag", "text"]
    ].head()
)

,title,genres,tag,text
0,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,pixar pixar fun,Movie title: Toy Story (1995). Genres: Adventu...
1,Jumanji (1995),Adventure|Children|Fantasy,fantasy magic board game Robin Williams game,Movie title: Jumanji (1995). Genres: Adventure...
2,Grumpier Old Men (1995),Comedy|Romance,moldy old,Movie title: Grumpier Old Men (1995). Genres: ...
3,Waiting to Exhale (1995),Comedy|Drama|Romance,,Movie title: Waiting to Exhale (1995). Genres:...
4,Father of the Bride Part II (1995),Comedy,pregnancy remake,Movie title: Father of the Bride Part II (1995...


In [55]:
movie_embeddings = embedding_model.encode(
    movies_text["text"].tolist(),
    show_progress_bar=True
)

print("Embedding shape:", movie_embeddings.shape)
print("Movie rows:", len(movies_text))
print("Embedding rows:", len(movie_embeddings))

Batches: 100%|██████████| 305/305 [06:33<00:00,  1.29s/it]

Embedding shape: (9742, 768)
Movie rows: 9742
Embedding rows: 9742


In [56]:
recommend_movies(
    "I want a serious, intelligent science fiction movie with space, mystery and a deep story.",
    top_n=10
)

,title,genres,similarity
8917,Star Trek Beyond (2016),Action|Adventure|Sci-Fi,0.589569
8159,Star Trek Into Darkness (2013),Action|Adventure|Sci-Fi|IMAX,0.581272
8376,Interstellar (2014),Sci-Fi|IMAX,0.574382
7018,Star Trek (2009),Action|Adventure|Sci-Fi|IMAX,0.570746
4132,Star Trek: Nemesis (2002),Action|Drama|Sci-Fi|Thriller,0.556873
7202,Planet 51 (2009),Adventure|Animation|Children|Comedy|Sci-Fi,0.547668
8879,The Martian (2015),Adventure|Drama|Sci-Fi,0.543910
8473,I Origins (2014),Drama|Sci-Fi,0.542461
1346,Lost in Space (1998),Action|Adventure|Sci-Fi,0.542427
9432,The Space Between Us (2016),Adventure|Sci-Fi,0.542161


In [57]:
preferences = get_preferences(
    "I want a serious, intelligent science fiction movie with space, mystery and a deep story, but not superheroes."
)

print(preferences)

filtered_candidates = filter_movies(preferences)

print("Candidates:", len(filtered_candidates))

display(
    filtered_candidates[
        ["movieId", "title", "genres"]
    ].head(20)
)

{'genres': ['Sci-Fi', 'Mystery'], 'moods': ['serious', 'intelligent'], 'themes': ['space', 'deep story'], 'exclude': ['superheroes']}
Candidates: 69


,movieId,title,genres
28,29,"City of Lost Children, The (Cité des enfants p...",Adventure|Drama|Fantasy|Mystery|Sci-Fi
31,32,Twelve Monkeys (a.k.a. 12 Monkeys) (1995),Mystery|Sci-Fi|Thriller
91,103,Unforgettable (1996),Mystery|Sci-Fi|Thriller
133,160,Congo (1995),Action|Adventure|Mystery|Sci-Fi
167,198,Strange Days (1995),Action|Crime|Drama|Mystery|Sci-Fi|Thriller
563,680,"Alphaville (Alphaville, une étrange aventure d...",Drama|Mystery|Romance|Sci-Fi|Thriller
932,1232,Stalker (1979),Drama|Mystery|Sci-Fi
1055,1372,Star Trek VI: The Undiscovered Country (1991),Action|Mystery|Sci-Fi
1391,1909,"X-Files: Fight the Future, The (1998)",Action|Crime|Mystery|Sci-Fi|Thriller
1484,2009,Soylent Green (1973),Drama|Mystery|Sci-Fi|Thriller


In [58]:
ranked_candidates = rank_candidates(
    "I want a serious, intelligent science fiction movie with space, mystery and a deep story.",
    filtered_candidates,
    top_n=10
)

display(ranked_candidates)

,movieId,title,genres,semantic_score
6331,48780,"Prestige, The (2006)",Drama|Mystery|Sci-Fi|Thriller,0.521915
7014,68237,Moon (2009),Drama|Mystery|Sci-Fi|Thriller,0.521008
8491,113350,I'll Follow You Down (2013),Drama|Mystery|Sci-Fi,0.519358
4650,6949,"Big Empty, The (2003)",Comedy|Mystery|Sci-Fi,0.499624
9668,182715,Annihilation (2018),Adventure|Mystery|Sci-Fi|Thriller,0.499454
9689,184253,The Cloverfield Paradox (2018),Horror|Mystery|Sci-Fi|Thriller,0.493000
8533,114935,Predestination (2014),Action|Mystery|Sci-Fi|Thriller,0.491235
3559,4874,K-PAX (2001),Drama|Fantasy|Mystery|Sci-Fi,0.488203
9262,156675,Project X (1968),Mystery|Sci-Fi,0.487840
6802,60760,"X-Files: I Want to Believe, The (2008)",Drama|Mystery|Sci-Fi|Thriller,0.487761


In [59]:
def gemini_rerank(user_query, candidates, top_n=10):

    movie_list = []

    for _, row in candidates.iterrows():
        movie_list.append({
            "movieId": int(row["movieId"]),
            "title": row["title"],
            "genres": row["genres"]
        })

    prompt = f"""
You are the final ranking system for a movie recommendation assistant.

User request:
"{user_query}"

Rank the following candidate movies from best match to worst match.

Consider:
- requested genres
- mood and tone
- themes
- story concepts
- settings
- exclusions
- overall fit to the user's request

Do NOT recommend movies that violate an explicit exclusion.

Return ONLY valid JSON in this exact format:

[
  {{
    "movieId": 123,
    "score": 95,
    "reason": "Short reason why this movie fits."
  }}
]

Rules:
- Include every candidate exactly once.
- score must be an integer from 0 to 100.
- Higher score means a better match.
- Do not invent movie information beyond the title and genres provided.
- Keep each reason under 20 words.

Candidates:
{movie_list}
"""

    response = client.models.generate_content(
        model="gemini-3.6-flash",
        contents=prompt
    )

    return json.loads(response.text)

In [60]:
top_20 = rank_candidates(
    "I want a serious, intelligent science fiction movie with space, mystery and a deep story.",
    filtered_candidates,
    top_n=20
)

gemini_results = gemini_rerank(
    "I want a serious, intelligent science fiction movie with space, mystery and a deep story.",
    top_20,
    top_n=10
)

gemini_results

[{'movieId': 68237,
  'score': 98,
  'reason': 'Serious, intelligent sci-fi mystery set on a lunar base with a deep story.'},
 {'movieId': 1372,
  'score': 85,
  'reason': 'Intelligent sci-fi political mystery set in space.'},
 {'movieId': 184253,
  'score': 78,
  'reason': 'Sci-fi mystery set aboard a space station in orbit.'},
 {'movieId': 182715,
  'score': 75,
  'reason': 'Serious, highly intelligent sci-fi mystery with a deep, philosophical story.'},
 {'movieId': 6303,
  'score': 73,
  'reason': 'Intelligent, serious hard sci-fi mystery involving an alien organism.'},
 {'movieId': 114935,
  'score': 70,
  'reason': 'Complex, intelligent sci-fi mystery with a deep narrative.'},
 {'movieId': 48780,
  'score': 68,
  'reason': 'Serious, intelligent mystery with sci-fi elements and deep storytelling.'},
 {'movieId': 85414,
  'score': 65,
  'reason': 'Intelligent, thought-provoking sci-fi mystery thriller.'},
 {'movieId': 5445,
  'score': 63,
  'reason': 'Intelligent futuristic sci-fi m

In [61]:
test_query = """
I want a psychological thriller that is dark, disturbing,
intelligent and has a major twist. No comedy or romance.
"""

preferences = get_preferences(test_query)

print(preferences)

filtered_candidates = filter_movies(preferences)

print("Candidates:", len(filtered_candidates))

top_20 = rank_candidates(
    test_query,
    filtered_candidates,
    top_n=20
)

gemini_results = gemini_rerank(
    test_query,
    top_20
)

gemini_results[:10]

{'genres': ['Thriller'], 'moods': ['dark', 'disturbing', 'intelligent'], 'themes': ['plot twist'], 'exclude': ['comedy', 'romance']}
Candidates: 1603


[{'movieId': 74458,
  'score': 98,
  'reason': 'A dark, intelligent psychological thriller featuring a disturbing atmosphere and a iconic major plot twist.'},
 {'movieId': 48780,
  'score': 95,
  'reason': 'An intelligent, dark psychological thriller filled with obsession, mystery, and a massive plot twist.'},
 {'movieId': 4848,
  'score': 92,
  'reason': 'A deeply disturbing, dark, and intelligent psychological mystery thriller with a mind-bending structure.'},
 {'movieId': 8957,
  'score': 90,
  'reason': 'A dark, disturbing psychological horror thriller renowned for its brilliant and shocking twist.'},
 {'movieId': 6214,
  'score': 87,
  'reason': 'An intensely dark, disturbing psychological crime drama and mystery with an unsettling narrative.'},
 {'movieId': 3535,
  'score': 84,
  'reason': 'A dark and disturbing psychological crime thriller exploring mind games and madness.'},
 {'movieId': 27904,
  'score': 80,
  'reason': 'An intelligent, paranoid psychological sci-fi mystery wi

In [62]:
def gemini_rerank(user_query, candidates, top_n=10):

    movie_list = []

    for _, row in candidates.iterrows():

        movie_tags = tags[
            tags["movieId"] == row["movieId"]
        ]["tag"].astype(str).tolist()

        movie_list.append({
            "movieId": int(row["movieId"]),
            "title": row["title"],
            "genres": row["genres"],
            "tags": movie_tags
        })

    prompt = f"""
You are the final ranking system for a movie recommendation assistant.

User request:
"{user_query}"

Rank these candidate movies according to how well they match the user's request.

Available movie information comes ONLY from MovieLens:
- title
- genres
- user-submitted tags

Use the user's requested:
- genres
- mood
- tone
- themes
- exclusions

Important grounding rules:
- Do NOT invent plot details.
- Do NOT claim a movie has a specific setting, character,
  plot twist, story element, or feature unless it is supported
  by the provided genres or tags.
- If the provided information is insufficient to justify a detail,
  do not mention that detail.
- Explicit exclusions must be respected.

Return ONLY valid JSON in exactly this format:

[
  {{
    "movieId": 123,
    "score": 95,
    "reason": "Short explanation based only on the provided information."
  }}
]

Rules:
- Include every candidate exactly once.
- score must be an integer from 0 to 100.
- Higher score means a better match.
- Keep each reason under 20 words.

Candidates:
{movie_list}
"""

    response = client.models.generate_content(
        model="gemini-3.6-flash",
        contents=prompt
    )

    return json.loads(response.text)

In [63]:
test_query = """
I want a psychological thriller that is dark, disturbing,
intelligent and has a major twist. No comedy or romance.
"""

top_20 = rank_candidates(
    test_query,
    filter_movies(get_preferences(test_query)),
    top_n=20
)

gemini_results = gemini_rerank(
    test_query,
    top_20
)

gemini_results[:10]

[{'movieId': 74458,
  'score': 98,
  'reason': 'Has tags for psychological thriller, plot twist, and thought-provoking, matching user requirements perfectly.'},
 {'movieId': 8957,
  'score': 95,
  'reason': 'Tagged with disturbing, clever, mindfuck, and surprise ending, strongly matching mood, intelligence, and twist requests.'},
 {'movieId': 6214,
  'score': 75,
  'reason': 'Tagged with dark and disturbing, matching the requested dark and disturbing tone for a mystery thriller.'},
 {'movieId': 48780,
  'score': 60,
  'reason': 'Drama mystery thriller tagged with atmospheric and enigmatic, matching general thriller and intelligence tone requests.'},
 {'movieId': 3535,
  'score': 40,
  'reason': 'Mystery horror thriller matching basic genre requests, but has no tags to verify specific tone or twist.'},
 {'movieId': 4848,
  'score': 38,
  'reason': 'Crime drama film-noir mystery thriller matching basic thriller genres, but lacks specific tags.'},
 {'movieId': 2389,
  'score': 35,
  'reas

In [64]:
import os
import joblib

os.makedirs("../models", exist_ok=True)

joblib.dump(
    movie_embeddings,
    "../models/movie_embeddings.pkl"
)

joblib.dump(
    user_factors,
    "../models/user_factors.pkl"
)

joblib.dump(
    movie_factors,
    "../models/movie_factors.pkl"
)

joblib.dump(
    user_movie_matrix,
    "../models/user_movie_matrix.pkl"
)

movies_text.to_pickle(
    "../models/movies_text.pkl"
)

print("All recommendation models and data saved successfully!")

All recommendation models and data saved successfully!


In [65]:
import os

for filename in os.listdir("../models"):
    path = os.path.join("../models", filename)
    size_mb = os.path.getsize(path) / (1024 * 1024)

    print(f"{filename}: {size_mb:.2f} MB")

movies_text.pkl: 1.98 MB
movie_embeddings.pkl: 28.54 MB
movie_factors.pkl: 3.71 MB
user_factors.pkl: 0.23 MB
user_movie_matrix.pkl: 45.33 MB
